# Clase 10. Rainbow y el aprendizaje distribucional: cuando varias mejoras se suman

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En la clase siete conociste Deep Q-Network (DQN): un agente que combina aprendizaje-Q con una red neuronal profunda, una red objetivo y repetición de experiencia para aprender directamente de la experiencia. Fue un hito, pero no fue un punto final. Desde su publicación, la comunidad investigadora en aprendizaje por refuerzo (RL) ha ido proponiendo mejoras concretas a distintas piezas de ese mismo algoritmo: cómo elige acciones, qué recuerda, cómo explora, qué predice. Hoy ves qué pasa cuando las juntas todas: el resultado se llama **Rainbow**.

## Guion de la clase

1. Rainbow: la suma de seis mejoras independientes
2. Qué reveló el estudio de ablación
3. Líneas de investigación activas que verás nombradas en la literatura
4. Código en vivo: añade una arquitectura dueling a tu DQN
5. Puntos clave y qué viene en la clase 11


## 1. Rainbow: la suma de seis mejoras independientes

*Guion de clase: dibuja un arcoíris en la pizarra con seis franjas y ve escribiendo el nombre de cada mejora sobre una franja a medida que la presentas.*

Un arcoíris surge cuando se combinan colores que ya existían de forma independiente. Del mismo modo, Rainbow aparece cuando juntas seis mejoras que ya se habían probado con éxito, cada una por su cuenta, sobre el DQN original. Ningún componente es exclusivo de Rainbow: lo que aporta el conjunto es la evidencia de que integrarlas todas en el mismo agente funciona mejor que cualquiera por separado.

Repásalas una a una:

- **Doble aprendizaje-Q**: separa la elección de la mejor acción (red online) de su evaluación (red objetivo), para reducir la sobreestimación típica de DQN
- **Repetición de experiencia priorizada (PER)**: en vez de muestrear transiciones al azar, repite con mayor frecuencia aquellas donde el agente se equivocó, porque son las que más información nueva aportan
- **Arquitectura dueling**: separa la predicción en dos preguntas: cuánto vale el estado en general —V(s)— y, dentro de ese estado, qué tan buena es cada acción frente a las demás —A(s,a)—, en vez de predecir un único valor Q por acción
- **Retornos multi-paso**: acumula la recompensa de varios pasos seguidos antes de bootstrapear con la red objetivo, en vez de mirar solo un paso hacia delante
- **Aprendizaje distribucional (C51)**: aprende la distribución completa de retornos posibles, no solo su promedio, representada mediante 51 puntos fijos (átomos) y comparada con la distribución objetivo mediante la divergencia de Kullback-Leibler (KL)
- **Redes ruidosas (Noisy Nets)**: sustituyen la exploración epsilon-greedy por ruido aprendible en los pesos de la red, para que sea la propia red quien decida cuánta exploración necesita en cada situación

Dos aclaraciones antes de seguir. Primera: la distribución que aprende C51 mide la variabilidad real de los resultados posibles, no la incertidumbre del agente. Segunda: el ejemplo visual que usa el material fuente para la arquitectura dueling es un videojuego de carreras, donde la cabeza de valor atiende a la escena completa (¿voy bien en la carrera?) mientras que la cabeza de ventaja se fija en detalles inmediatos, como el coche que tienes al lado.


## 2. Qué reveló el estudio de ablación

*Guion de clase: pregunta antes de mostrar la respuesta — "¿qué mejora creéis que aporta menos una vez combinada con el resto?"*

Cuando el equipo que propuso Rainbow probó qué pasaba al retirar cada componente uno por uno, el resultado no fue simétrico:

- El aprendizaje multi-paso, la repetición priorizada y el aprendizaje distribucional resultaron ser los que más sumaban al rendimiento final
- El doble aprendizaje-Q y la arquitectura dueling aportaron menos de lo esperado una vez combinados con el resto: una explicación propuesta es que el aprendizaje distribucional ya limita por sí mismo el rango de retornos posibles, lo que reduce buena parte del problema de sobreestimación que el doble aprendizaje-Q había sido diseñado para resolver

En eficiencia de datos, la cifra que se suele citar es contundente: Rainbow logra, en unos siete millones de fotogramas de entrenamiento, la misma puntuación media normalizada que el DQN estándar necesita cerca de 200 millones de fotogramas para conseguir. Es una mejora de eficiencia de datos de aproximadamente 14 veces, con un tiempo de cómputo similar al del DQN original.


## 3. Líneas de investigación activas que verás nombradas en la literatura

Más allá de Rainbow, encontrarás otros nombres que circulan con frecuencia en RL avanzado. El material de referencia de esta clase solo los menciona de pasada, así que trátalos como mapa de lo que existe, no como contenido desarrollado aquí:

- **RL multiagente**: varios agentes que aprenden a la vez en el mismo entorno, a veces cooperando y a veces compitiendo; conceptos habituales incluyen el entrenamiento centralizado con ejecución descentralizada y el modelado del comportamiento de otros agentes
- **RL jerárquico**: el agente aprende también acciones extendidas en el tiempo, llamadas opciones, cada una con su propia política interna y su propia condición de terminación, en vez de aprender solo movimientos de un paso
- **Meta-aprendizaje, aprendizaje por transferencia y aprendizaje por currículo**: formas de que un agente aproveche lo aprendido en una tarea para resolver más rápido otra tarea relacionada, o de organizar el entrenamiento en un orden de dificultad creciente
- **Seguridad en RL**: un área que gana relevancia a medida que estos agentes se acercan a entornos reales; el aprendizaje distribucional, por ejemplo, ya aporta algo útil aquí, porque informa sobre la probabilidad de terminaciones tempranas o indeseadas, y no solo sobre el retorno esperado

**Si alguno de estos nombres te resulta interesante, apúntalo como línea propia de estudio.**


## 4. Código en vivo: añade una arquitectura dueling a tu DQN

*Guion de clase: antes de tocar código, recupera en voz alta las piezas del DQN de la clase siete: red online, red objetivo, buffer de repetición y exploración epsilon-greedy. Hoy cambias solo una pieza: la arquitectura interna de la red.*

Programar las seis mejoras de Rainbow en una sola clase no sería realista. En cambio, vas a retomar el DQN mínimo de la clase siete y añadirle una sola mejora, bien delimitada: la arquitectura dueling. Vas a entrenar dos agentes sobre CartPole (un DQN estándar y un Dueling DQN) con exactamente el mismo bucle de entrenamiento, y vas a comparar sus curvas de recompensa. La única diferencia entre ambos va a ser la arquitectura interna de la red.

**Ejecuta cada celda en orden.** Están divididas en pasos pequeños para que puedas parar, comentar el resultado y responder preguntas antes de seguir.


In [ ]:
!pip install gymnasium


### Preparamos el entorno de trabajo

Importamos gymnasium (la librería estándar de entornos de RL), PyTorch para la red neuronal, y fijamos una semilla para que el experimento sea reproducible.


In [ ]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Fijamos semillas para que el experimento sea reproducible
SEMILLA = 0
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)


### El buffer de repetición de experiencia

*Guion de clase: recuerda que esta pieza ya la construiste en la clase siete — hoy no cambia.*

`BufferReplay` guarda transiciones pasadas y permite muestrear lotes al azar, rompiendo la correlación temporal de los datos consecutivos de un episodio. Es idéntico al de la clase siete.


In [ ]:
Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferReplay:
    """Memoria de repeticion de experiencia (clase siete): guarda transiciones
    pasadas y permite muestrear lotes al azar, rompiendo la correlacion
    temporal de los datos consecutivos de un episodio."""

    def __init__(self, capacidad=10000):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


### La red del DQN estándar

Una sola cabeza que produce directamente un valor Q por cada acción posible. Es la arquitectura "de manual" que ya viste en la clase siete.


In [ ]:
class RedQEstandar(nn.Module):
    """DQN "de manual": una sola cabeza que produce directamente
    un valor Q por cada accion posible."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        return self.red(x)


### La red del Dueling DQN

*Guion de clase: dibuja el tronco compartido y las dos cabezas en la pizarra antes de mostrar el código. Insiste en la fórmula de combinación: por qué restamos la media de la ventaja.*

Un tronco compartido alimenta dos cabezas: una estima V(s) (cuánto vale el estado, un único número) y otra estima A(s,a) (cuánto mejor es cada acción frente a las demás). Se combinan restando la media de la ventaja, para que la suma quede en una escala estable:

**Q(s,a) = V(s) + (A(s,a) − media_a[A(s,a)])**


In [ ]:
class RedQDueling(nn.Module):
    """Arquitectura dueling: un tronco compartido alimenta dos cabezas.
    Una estima V(s) -- cuanto vale el estado, un unico numero --
    y otra estima A(s,a) -- cuanto mejor es cada accion frente a las demas.
    Se combinan restando la media de la ventaja, para que la suma
    quede en una escala estable."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
        )
        self.cabeza_valor = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, 1),
        )
        self.cabeza_ventaja = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        caracteristicas = self.tronco(x)
        valor = self.cabeza_valor(caracteristicas)
        ventaja = self.cabeza_ventaja(caracteristicas)
        # Q(s,a) = V(s) + (A(s,a) - media_a[A(s,a)])
        # dim=-1 funciona igual con un estado suelto (sin lote) que con un
        # lote de estados: siempre promedia sobre el eje de las acciones.
        return valor + (ventaja - ventaja.mean(dim=-1, keepdim=True))


### El bucle de entrenamiento

*Guion de clase: subraya que el bucle es exactamente el mismo para las dos redes. La única diferencia entre ambos experimentos es qué clase de red le pasas a `entrenar`.*

`entrenar` recibe la clase de red como parámetro. El resto del algoritmo —red online, red objetivo, buffer de repetición, exploración epsilon-greedy decreciente— no cambia entre experimentos.


In [ ]:
def entrenar(clase_red, nombre, n_episodios=120, gamma=0.99, tamano_lote=64,
             tasa_aprendizaje=1e-3, actualizar_objetivo_cada=10):
    """Bucle de entrenamiento DQN estandar: red online, red objetivo,
    buffer de repeticion y exploracion epsilon-greedy decreciente.
    Cambiar `clase_red` entre RedQEstandar y RedQDueling es la unica
    diferencia entre ambos experimentos -- el resto del algoritmo no cambia."""

    entorno = gym.make("CartPole-v1")
    n_entradas = entorno.observation_space.shape[0]
    n_acciones = entorno.action_space.n

    red_online = clase_red(n_entradas, n_acciones)
    red_objetivo = clase_red(n_entradas, n_acciones)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=tasa_aprendizaje)
    buffer = BufferReplay(capacidad=10000)

    epsilon, epsilon_minimo, decaimiento = 1.0, 0.05, 0.97
    recompensas_por_episodio = []

    for episodio in range(n_episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_total = 0.0
        terminado = False

        while not terminado:
            if random.random() < epsilon:
                accion = entorno.action_space.sample()  # exploracion
            else:
                with torch.no_grad():
                    q_valores = red_online(torch.tensor(estado, dtype=torch.float32))
                    accion = int(torch.argmax(q_valores).item())  # explotacion

            siguiente_estado, recompensa, fin, truncado, _ = entorno.step(accion)
            terminado = fin or truncado
            buffer.guardar(estado, accion, recompensa, siguiente_estado, fin)
            estado = siguiente_estado
            recompensa_total += recompensa

            # Solo entrenamos cuando el buffer tiene datos suficientes
            if len(buffer) >= tamano_lote:
                lote = buffer.muestrear(tamano_lote)
                estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
                acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64)
                recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
                siguientes = torch.tensor(np.array([t.siguiente_estado for t in lote]), dtype=torch.float32)
                terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

                q_actuales = red_online(estados).gather(1, acciones.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_siguientes = red_objetivo(siguientes).max(dim=1)[0]
                    objetivo = recompensas + gamma * q_siguientes * (1 - terminados)

                perdida = nn.functional.mse_loss(q_actuales, objetivo)
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

        recompensas_por_episodio.append(recompensa_total)
        epsilon = max(epsilon_minimo, epsilon * decaimiento)

        if episodio % actualizar_objetivo_cada == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    entorno.close()
    print(f"{nombre}: recompensa media de los ultimos 20 episodios = "
          f"{np.mean(recompensas_por_episodio[-20:]):.1f}")
    return recompensas_por_episodio


### Entrenamos ambos agentes

**Ejecuta esta celda y aprovecha el tiempo de espera (dura entre uno y dos minutos en Colab) para repasar en voz alta las cuatro piezas del bucle.**


In [ ]:
# --- Entrenamos ambos agentes con el mismo bucle ---
recompensas_dqn = entrenar(RedQEstandar, "DQN estandar")
recompensas_dueling = entrenar(RedQDueling, "Dueling DQN")


### Comparamos las curvas de recompensa

Suavizamos cada curva con una media móvil de diez episodios, para que la tendencia se vea sin el ruido de episodio a episodio.


In [ ]:
def media_movil(valores, ventana=10):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")


# --- Comparamos las curvas de recompensa ---
plt.figure(figsize=(10, 5))
plt.plot(media_movil(recompensas_dqn), label="DQN estandar")
plt.plot(media_movil(recompensas_dueling), label="Dueling DQN")
plt.xlabel("Episodio")
plt.ylabel("Recompensa (media movil de 10 episodios)")
plt.title("CartPole-v1: DQN estandar vs. Dueling DQN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Lectura del resultado**

*Guion de clase: para en esta gráfica y pregunta a la clase qué curva esperaban y cuál ven. No fuerces una conclusión: en CartPole la ventaja del Dueling DQN suele ser modesta, y qué red gana puede variar entre ejecuciones.*

Fíjate en dos cosas: cuánto tarda cada curva en acercarse a la recompensa máxima de CartPole (500 pasos) y cuánta variabilidad muestra cada una entre episodios. En un entorno tan simple como CartPole, la ventaja del Dueling DQN suele ser modesta: la arquitectura dueling brilla más en entornos con muchos estados donde la acción elegida casi no cambia el resultado. Lo que sí te llevas de este ejercicio es el esqueleto completo para añadir, más adelante, las otras cinco mejoras de Rainbow sobre esta misma base.


## Puntos clave

- DQN, de la clase siete, es el punto de partida; Rainbow es la suma de seis mejoras independientes que, combinadas, superan claramente a DQN por separado
- Las seis mejoras son: doble aprendizaje-Q, repetición de experiencia priorizada, arquitectura dueling, retornos multi-paso, aprendizaje distribucional (C51) y redes ruidosas
- El estudio de ablación muestra que multi-paso, repetición priorizada y aprendizaje distribucional aportan más que doble aprendizaje-Q y dueling una vez combinados
- Rainbow logra en unos siete millones de fotogramas lo que DQN tarda cerca de 200 millones en conseguir: aproximadamente 14 veces más eficiencia de datos
- El aprendizaje distribucional aprende la distribución completa de retornos posibles, no solo su media, y usa la divergencia KL como función de pérdida
- RL multiagente, RL jerárquico, meta-aprendizaje/transferencia/currículo y seguridad en RL son líneas de investigación activas que verás nombradas en la literatura, aunque esta clase no las desarrolla en profundidad

## Antes de la próxima clase

**Abre el notebook de práctica de esta clase y completa los dos ejercicios: la arquitectura dueling desde cero y los retornos multi-paso.**

En la clase 11 cambias de perspectiva: en vez de mejorar un único agente que aprende solo, vas a ver arquitecturas donde varias unidades de cómputo entrenan en paralelo, como A3C e IMPALA.
